# Phase 6: Stronger Ranking Models and Failure Reduction

In [2]:
import sys
! {sys.executable} -m pip install lightgbm

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ndcg_score
import lightgbm as lgb
import json
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

PROJECT_ROOT=Path.cwd().parent
DATA_RAW=PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED=PROJECT_ROOT / 'data' / 'processed'
PHASE5_OUTPUT=DATA_PROCESSED / 'phase5_failure_analysis'
PHASE6_OUTPUT=DATA_PROCESSED / 'phase6_models'
PHASE6_OUTPUT.mkdir(parents=True, exist_ok=True)

print(f"Phase 6 output: {PHASE6_OUTPUT}")

PIPELINES=['raw', 'global', 'per_query']
K_VALUES=[1, 3, 5, 10]
RANDOM_SEED=36
MAX_PAIRS_PER_QUERY=1000

Phase 6 output: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase6_models


## 1. Data Loading

In [4]:
print("="*80)
print("DATA LOADING")
print("="*80)

def load_letor_fold(dataset_name, fold_num):
    base_path=DATA_RAW / dataset_name / f'Fold{fold_num}'
    splits={}
    
    for split_name in ['train', 'vali', 'test']:
        file_path=base_path / f'{split_name}.txt'
        if not file_path.exists():
            raise RuntimeError(f"Missing: {file_path}")
        
        data=[]
        with open(file_path, 'r') as f:
            for line in f:
                if '#' in line:
                    line=line[:line.index('#')]
                parts=line.strip().split()
                if len(parts)<2:
                    continue
                
                label=int(parts[0])
                qid=int(parts[1].split(':')[1])
                features={}
                for item in parts[2:]:
                    if ':' in item:
                        fid, fval=item.split(':')
                        fid=int(fid)
                        if 1<=fid<=46:
                            features[f'f{fid}']=float(fval)
                row={'qid': qid, 'label': label}
                row.update(features)
                data.append(row)
        
        df=pd.DataFrame(data)
        for i in range(1, 47):
            if f'f{i}' not in df.columns:
                df[f'f{i}']=0.0
        df=df.drop(columns=[f'f{i}' for i in range(6, 11)], errors='ignore')
        splits[split_name]=df
        print(f"{dataset_name} {split_name:5s}: {len(df):6d} docs, {df['qid'].nunique():4d} queries")
    
    return splits['train'], splits['vali'], splits['test']

print("\nMQ2007:")
train_2007, vali_2007, test_2007=load_letor_fold('MQ2007', 1)
print("\nMQ2008:")
_, _, test_2008=load_letor_fold('MQ2008', 1)

feature_cols=[c for c in train_2007.columns if c.startswith('f')]
print(f"\nFeatures: {len(feature_cols)}")
print("DATA LOADED")

DATA LOADING

MQ2007:
MQ2007 train:  42158 docs, 1017 queries
MQ2007 vali :  13813 docs,  339 queries
MQ2007 test :  13652 docs,  336 queries

MQ2008:
MQ2008 train:   9630 docs,  471 queries
MQ2008 vali :   2707 docs,  157 queries
MQ2008 test :   2874 docs,  156 queries

Features: 41
DATA LOADED


## 2. Metrics

In [5]:
def precision_at_k(ranked_labels, k, relevance_threshold=1):
    if k<=0:
        raise ValueError("k must be positive")
    if relevance_threshold not in [1, 2]:
        raise ValueError("relevance_threshold must be 1 or 2")
    k=min(k, len(ranked_labels))
    if k==0:
        return 0.0
    top_k=ranked_labels[:k]
    relevant=sum(1 for label in top_k if label >= relevance_threshold)
    return relevant / k

def ndcg_at_k(ranked_labels, k):
    if k<=0:
        raise ValueError("k must be positive")
    k=min(k, len(ranked_labels))
    if k==0:
        return 0.0
    
    def dcg(labels, k):
        labels_k=labels[:k]
        gains=[2**label - 1 for label in labels_k]
        discounts=[np.log2(i + 2) for i in range(len(labels_k))]
        return sum(g / d for g, d in zip(gains, discounts))
    
    dcg_val=dcg(ranked_labels, k)
    ideal=sorted(ranked_labels, reverse=True)
    idcg_val=dcg(ideal, k)
    return dcg_val / idcg_val if idcg_val > 0 else 0.0

def failure_at_k(ranked_labels, k, relevance_threshold=1):
    """Empty query returns 0.0 (not evaluable)."""
    if k<=0:
        raise ValueError("k must be positive")
    if relevance_threshold not in [1, 2]:
        raise ValueError("relevance_threshold must be 1 or 2")
    k=min(k, len(ranked_labels))
    if k==0:
        return 0.0  
    top_k=ranked_labels[:k]
    has_relevant=any(label >= relevance_threshold for label in top_k)
    return 0.0 if has_relevant else 1.0

print("Metrics defined")

Metrics defined


## 3. NDCG Verification

In [6]:
print("="*80)
print("NDCG VERIFICATION ")
print("="*80)

sample_qids=test_2007['qid'].unique()[:5]
print(f"\n{'QID':<8s} {'Custom':<10s} {'sklearn':<10s} {'Diff':<12s}")
print("-"*45)

for qid in sample_qids:
    query_docs=test_2007[test_2007['qid']==qid].copy()
    n_docs=len(query_docs)
    
    if n_docs==0:
        continue
    
    #Creating deterministic scores in original order
    query_docs=query_docs.reset_index(drop=True)
    tmp_scores=np.linspace(1.0, 0.0, n_docs)
    query_docs['tmp_score']=tmp_scores
    
    k=min(5, n_docs)
    if k==0:
        continue
    
    #Custom NDCG: ranking by score, computing on ranked labels
    sorted_docs=query_docs.sort_values('tmp_score', ascending=False, kind='stable')
    ranked_labels=sorted_docs['label'].values
    custom=ndcg_at_k(ranked_labels, k)
    
    #sklearn NDCG: using original order 
    original_labels=query_docs['label'].values
    original_scores=query_docs['tmp_score'].values
    y_true_gains=np.array([[2**label - 1 for label in original_labels]])
    y_score=original_scores.reshape(1, -1)
    sklearn_ndcg=ndcg_score(y_true_gains, y_score, k=k)
    
    diff=abs(custom - sklearn_ndcg)
    print(f"{qid:<8d} {custom:<10.6f} {sklearn_ndcg:<10.6f} {diff:<12.9f}")
    
    if diff>1e-6:
        print(f"WARNING: diff={diff:.2e}")

print("\n NDCG verified ")
print("="*80)

NDCG VERIFICATION 

QID      Custom     sklearn    Diff        
---------------------------------------------
7968     0.214533   0.214533   0.000000000 
7979     0.000000   0.000000   0.000000000 
7993     0.131205   0.131205   0.000000000 
7995     0.195190   0.195190   0.000000000 
8002     0.277273   0.277273   0.000000000 

 NDCG verified 


We are just checking if our ndcg calculation is correct. For a few example queries, it creates simple fake scores so that the documents are ranked in a known order. Then it uses our ndcg function and secondly, uses sklearn's built-in ndcg_score

For sklearn, we give the original labels and the fake scores, and let sklearn figure out the ranking by itself. Then comparing the two results, we see that both numbers are the same.